# 04 — Federated districting: an interactive POC

Districts are found by **partitioning the network's own graph** (contiguous by
construction, unlike clustering a dense pairwise matrix) with edge weights combining
three tunable factors:

1. **Structural cut cost** — pipe resistance ($L/D^5$) as baseline connection strength,
   valves and closed links down-weighted as natural cut candidates. Always available, no
   hydraulic solve, works on any network.
2. **Elevation similarity** — gravity systems and PRV placement track elevation bands.
   Checked per network before use: a flat network (Graeme, span 0 m) makes this factor
   inert regardless of its weight, and that's verified rather than assumed.
3. **Measured hydraulic coupling** (optional) — if an interventional map from notebook 02
   exists for this network, its symmetrised sensitivity is blended in. Its coverage is
   reported explicitly: the map is a probe-by-responder matrix, typically a subset of
   nodes, so it can only inform edges where *both* endpoints were probed.

Balance (comparable client data volume) is a bounded post-hoc local search, not a hard
constraint in the objective — a balance constraint can force a cut through a poorly
separable region, which is precisely the wrong trade for a federated client boundary.

**The central discipline of this notebook:** the cheap structural proxy is validated
against real hydraulics wherever ground truth exists, never assumed correct. It is
validated twice — against D-Town's actual published 5-DMA partition, and against
Graeme's measured near-total coupling — and the second validation is a warning, not a
pass: on Graeme the proxy said 0.05 (looks separable) while the true ratio is 0.78 (isn't).
Both are shown below before the interactive part.

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wntr

sys.path.insert(0, str(Path.cwd()))
import wdn_dependency as dep
import wdn_districting as di

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 200)
warnings.simplefilter('ignore')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})


def find_network_dir(candidates=('../../data/00_networks', 'nets', 'networks', '.')):
    """Locate the corpus without hardcoding a path, so the notebook is portable."""
    for c in candidates:
        p = Path(c)
        if p.is_dir() and any(p.glob('*.inp')):
            return p
    raise FileNotFoundError('no .inp files found under any of %s' % (candidates,))



NETWORK_DIR = find_network_dir()
GT_DIR = Path('data/08_reporting/dependency/ground_truth')
OUT_DIR = Path('data/08_reporting/districting')
OUT_DIR.mkdir(parents=True, exist_ok=True)
NETWORKS = sorted(p.stem for p in NETWORK_DIR.glob('*.inp'))
# p.stem only strips the .parquet extension, leaving "Graeme__coupling" -- the
# "__coupling" suffix must be removed explicitly or every membership check
# against a bare network name (e.g. "Graeme" in HAS_COUPLING) silently fails.
HAS_COUPLING = sorted(p.stem.replace('__coupling', '')
                      for p in GT_DIR.glob('*__coupling.parquet')) \
    if GT_DIR.exists() else []
print('%d networks; ground-truth coupling available for: %s'
      % (len(NETWORKS), HAS_COUPLING or 'none (run notebook 02 first for validation)'))


26 networks; ground-truth coupling available for: ['Balerma', 'Graeme', 'ky1', 'ky14', 'ky16', 'ky17', 'ky18', 'ky2', 'ky3', 'ky4', 'ky7', 'temp']


## 1. Validation A — D-Town's real published DMA partition

D-Town's `.inp` carries an actual utility districting decision in `[TAGS]` (5 DMAs). This
is the one external, non-synthetic ground truth in the corpus, so it is the strongest
available check: does the method recover a partition a human engineer actually chose?

In [48]:
wn_dt = wntr.network.WaterNetworkModel(str(NETWORK_DIR / 'D_Town.inp'))
for node in wn_dt.node_name_list:
    if isinstance(wn_dt.get_node(node), wntr.network.Junction):
        tag = wn_dt.get_node(node).demand_pattern.split('_')[0]
        wn_dt.get_node(node).tag = tag

gt_dma = di.load_dtown_dma_ground_truth(wn_dt)
print('%d nodes carry a real DMA tag across %d DMAs'
      % (len(gt_dma), len(set(gt_dma.values()))))

rows = []
for label, w in [
    ('resistance only', di.DistrictingWeights(1, 0, 0, 0)),
    ('elevation only', di.DistrictingWeights(0, 1, 0, 0)),
    ('resistance + elevation', di.DistrictingWeights(1, 1, 0, 0)),
    ('+ valve-respect (all structural factors)', di.DistrictingWeights(1, 0, 1, 0)),
]:
    out = di.districts_from_graph(wn_dt, k=5, weights=w, seed=3)
    ext = di.external_agreement(out['labels'], gt_dma)
    con = di.contiguity_report(out['graph'], out['labels'])
    rows.append({'factors': label, 'ami_vs_real_dma': round(ext['ami'], 4),
                 'ari_vs_real_dma': round(ext['ari'], 4),
                 'all_contiguous': bool(con.contiguous.all())})
display(pd.DataFrame(rows))
print("Elevation alone already recovers most of the structure (D-Town's DMAs are "
      "elevation-organised, as is typical); explicitly respecting the network's own "
      "5 valves adds the final increment. Raw resistance alone is markedly weaker.")


348 nodes carry a real DMA tag across 5 DMAs


,factors,ami_vs_real_dma,ari_vs_real_dma,all_contiguous
0,resistance only,0.8012,0.7930,True
1,elevation only,0.8571,0.8313,True
2,resistance + elevation,0.8571,0.8313,True
3,+ valve-respect (all structural factors),0.9462,0.9493,True


Elevation alone already recovers most of the structure (D-Town's DMAs are elevation-organised, as is typical); explicitly respecting the network's own 5 valves adds the final increment. Raw resistance alone is markedly weaker.


In [62]:
result = out['consumer_labels'].groupby(out['consumer_labels']).apply(lambda x: x.index.tolist()).to_dict()
result

{0: ['J411',
  'J414',
  'J417',
  'J314',
  'J315',
  'J316',
  'J210',
  'J211',
  'J212',
  'J214',
  'J217',
  'J218',
  'J110',
  'J421',
  'J1153',
  'J1154',
  'J1155',
  'J1157',
  'J1158',
  'J428',
  'J429',
  'J1056',
  'J1058',
  'J225',
  'J226',
  'J1160',
  'J1161',
  'J431',
  'J432',
  'J433',
  'J434',
  'J435',
  'J436',
  'J438',
  'J439',
  'J332',
  'J333',
  'J334',
  'J335',
  'J336',
  'J337',
  'J444',
  'J341',
  'J142',
  'J143',
  'J144',
  'J154',
  'J155',
  'J156',
  'J159',
  'J365',
  'J366',
  'J367',
  'J369',
  'J160',
  'J161',
  'J162',
  'J163',
  'J164',
  'J165',
  'J166',
  'J167',
  'J95',
  'J96',
  'J97',
  'J976',
  'J370',
  'J372',
  'J373',
  'J374',
  'J375',
  'J376',
  'J377',
  'J1219',
  'J379',
  'J171',
  'J172',
  'J173',
  'J174',
  'J175',
  'J177',
  'J179',
  'J1223',
  'J180',
  'J181',
  'J1024',
  'J183',
  'J1025',
  'J186',
  'J187',
  'J188',
  'J189',
  'J408',
  'J191',
  'J192',
  'J193',
  'J194',
  'J195',
  'J196

## 2. Validation B — the Graeme warning

Graeme's coupling is known from notebook 02 to be near-uniform (true coupling ratio 0.78,
essentially no separable structure). This section checks whether the cheap structural proxy
knows that, and what the honest ceiling is when the real coupling matrix is used directly.

In [3]:
if 'Graeme' in HAS_COUPLING:
    wn_g = wntr.network.WaterNetworkModel(str(NETWORK_DIR / 'Graeme.inp'))
    coupling_g = di.load_hydraulic_coupling('Graeme', GT_DIR)
    out_g = di.districts_from_graph(wn_g, k=5, weights=di.DistrictingWeights(1, 1, 1, 0), seed=1)

    proxy = di.structural_coupling_ratio(out_g['graph'], out_g['labels'])['structural_coupling_ratio']
    common = [n for n in coupling_g.index if n in coupling_g.columns]
    true_labels = out_g['labels'].reindex(common).dropna().astype(int)
    true_ratio = dep.district_coupling(coupling_g, true_labels)['coupling_ratio']
    cov = di.coupling_edge_coverage(out_g['graph'], coupling_g)
    ceiling = di.coupling_ceiling(coupling_g, 5, seed=1)

    print('cheap structural proxy says:      %.4f  (looks separable)' % proxy)
    print('TRUE hydraulic coupling ratio:    %.4f  (is NOT separable)' % true_ratio)
    print('coupling-matrix edge coverage:    %.1f%% of edges (%d probed of %d nodes)'
          % (100 * cov['coverage'], len(common), out_g['graph'].number_of_nodes()))
    print('ceiling (cluster on real S directly, best case for ANY method): %.4f'
          % ceiling['coupling_ratio'])
    print()
    print('The ceiling is close to the graph partition''s true ratio: no method -- graph-only, '
          'coupling-driven, or blended -- finds separable districts on Graeme, because the '
          'network genuinely has none. The lesson is not "use a better algorithm"; it is '
          '"validate the proxy before trusting it, and prefer a network with real structure '
          'to demonstrate the method."')
else:
    print('Graeme ground truth not found under', GT_DIR, '-- run notebook 02 first.')


cheap structural proxy says:      0.0537  (looks separable)
TRUE hydraulic coupling ratio:    0.7843  (is NOT separable)
coupling-matrix edge coverage:    25.6% of edges (60 probed of 114 nodes)
ceiling (cluster on real S directly, best case for ANY method): 0.7849

The ceiling is close to the graph partitions true ratio: no method -- graph-only, coupling-driven, or blended -- finds separable districts on Graeme, because the network genuinely has none. The lesson is not "use a better algorithm"; it is "validate the proxy before trusting it, and prefer a network with real structure to demonstrate the method."


## 3. Corpus scan

Elevation informativeness and valve/PRV counts per network, so a sensible default weight
combination can be picked per network before the interactive section — and so networks
known to be poorly separable (transmission-scale, single pressure zone) are flagged rather
than discovered by surprise.

In [4]:
def quick_profile(path):
    wn = wntr.network.WaterNetworkModel(str(path))
    g = di.build_feature_graph(wn)
    ec = di.check_elevation_informative(g)
    cons = dep.consumer_nodes(wn)
    return {'network': path.stem, 'n_nodes': g.number_of_nodes(),
           'n_consumers': len(cons), 'n_valves': len(wn.valve_name_list),
           'n_regulating_valves': sum(1 for v in wn.valve_name_list
                                      if wn.get_link(v).valve_type in di.VALVE_TYPES_REGULATING),
           'elevation_span_m': round(ec['span_m'], 1), 'elevation_informative': ec['informative'],
           'has_ground_truth': path.stem in HAS_COUPLING}


profile = pd.DataFrame([quick_profile(p) for p in NETWORK_DIR.glob('*.inp')]) \
    .set_index('network').sort_values('elevation_span_m')
profile.to_csv(OUT_DIR / 'corpus_districting_profile.csv')
profile

,n_nodes,n_consumers,n_valves,n_regulating_valves,elevation_span_m,elevation_informative,has_ground_truth
network,,,,,,,
Graeme,114,112,0,0,0.0,False,True
ky2,815,757,0,0,29.3,True,True
NJ1,14999,13306,0,0,37.0,True,False
ky3,275,249,0,0,43.0,True,True
ky18,776,495,2,0,53.2,True,True
ky14,384,331,0,0,64.7,True,True
ky7,485,463,0,0,70.0,True,True
ky4,964,934,0,0,75.0,True,True
ky5,427,392,0,0,75.1,True,False


## 4. Interactive: pick a network, factors and K, see the result

Sliders update the partition live. Metrics shown: node-count/demand balance, contiguity
(should always read all-True — it's enforced, not hoped for), elevation separation, the
cheap structural coupling ratio, and — whenever ground truth exists for the selected
network — the TRUE hydraulic coupling ratio and the coupling ceiling, so the cheap proxy
is checked every time it's used rather than trusted by default.

In [47]:
import ipywidgets as widgets
from IPython.display import display as ip_display

_cache = {}


def get_wn(network):
    if network not in _cache:
        _cache[network] = wntr.network.WaterNetworkModel(str(NETWORK_DIR / (network + '.inp')))
    return _cache[network]


def get_coupling(network):
    key = ('coupling', network)
    if key not in _cache:
        _cache[key] = di.load_hydraulic_coupling(network, GT_DIR)
    return _cache[key]


def render(network='D_Town', k=5, w_resistance=1.0, w_elevation=1.0, w_valve_cut=1.0,
          w_coupling=0.0, balance_target='count', balance_tol=0.15, seed=3):
    wn = get_wn(network)
    coupling = get_coupling(network) if w_coupling > 0 else None
    weights = di.DistrictingWeights(w_resistance, w_elevation, w_valve_cut, w_coupling)
    bt = None if balance_target == 'none' else balance_target

    out = di.districts_from_graph(wn, k=k, weights=weights, coupling=coupling,
                                  balance_target=bt, balance_tol=balance_tol, seed=seed)
    rep = di.full_report(out['graph'], out['labels'], out['consumer_labels'])
    con = rep['contiguity']

    print('%s: requested K=%d, achieved %d districts, %d fragment(s) repaired'
          % (network, k, out['n_districts_final'], out['n_fragments_repaired']))
    if not out['elevation_check']['informative'] and w_elevation > 0:
        print('WARNING: elevation span is %.1f m on this network -- the elevation factor '
              'is inert here regardless of its weight.' % out['elevation_check']['span_m'])
    print('structural coupling ratio (cheap proxy): %.4f'
          % rep['structural']['structural_coupling_ratio'])

    truth_line = 'no ground truth available for this network'
    ground_truth_gt = di.load_dtown_dma_ground_truth(wn) if network == 'D_Town' else None
    if network in HAS_COUPLING:
        real_S = get_coupling(network)
        common = [n for n in real_S.index if n in real_S.columns]
        lab_common = out['labels'].reindex(common).dropna().astype(int)
        if len(lab_common) >= 4:
            true_ratio = dep.district_coupling(real_S, lab_common)['coupling_ratio']
            cov = di.coupling_edge_coverage(out['graph'], real_S)
            truth_line = ('TRUE hydraulic coupling ratio: %.4f  (probe coverage: %.0f%% of edges)'
                          % (true_ratio, 100 * cov['coverage']))
    elif ground_truth_gt:
        ext = di.external_agreement(out['labels'], ground_truth_gt)
        truth_line = 'AMI vs real DMA tags: %.4f  (ARI %.4f, n=%d)' \
            % (ext['ami'], ext['ari'], ext['n_common'])
    print(truth_line)

    print('balance: count imbalance %.2fx, demand imbalance %.2fx'
          % (rep['balance']['count_imbalance'], rep['balance'].get('demand_imbalance', np.nan)))
    print('contiguity: %s (all districts, %d total)'
          % ('ALL CONTIGUOUS' if con.contiguous.all() else 'FRAGMENTED -- investigate', len(con)))
    print('elevation separation F-ratio: %.2f  (1 = no separation, higher = districts '
          'track elevation bands)' % rep['elevation']['f_ratio'])

    fig, ax = plt.subplots(figsize=(6.5, 6))
    for lname in wn.pipe_name_list:
        l = wn.get_link(lname)
        c1 = wn.get_node(l.start_node_name).coordinates
        c2 = wn.get_node(l.end_node_name).coordinates
        if c1 and c2:
            ax.plot([c1[0], c2[0]], [c1[1], c2[1]], lw=0.3, color='0.8', zorder=1)
    labs = out['labels']
    pts = np.array([wn.get_node(n).coordinates or (np.nan, np.nan) for n in labs.index],
                   dtype=float)
    sc = ax.scatter(pts[:, 0], pts[:, 1], c=labs.to_numpy(), cmap='tab10', s=22, zorder=3)
    for lname in wn.valve_name_list:
        v = wn.get_link(lname)
        c1 = wn.get_node(v.start_node_name).coordinates
        c2 = wn.get_node(v.end_node_name).coordinates
        if c1 and c2:
            ax.plot(*zip(c1, c2), color='red', lw=2.2, zorder=4)
    ax.set_title('%s — %d districts (red = valve links)' % (network, out['n_districts_final']))
    ax.set_aspect('equal')
    ax.axis('off')
    plt.show()
    return out


_ = widgets.interact(
    render,
    network=widgets.Dropdown(options=NETWORKS, value='D_Town'),
    k=widgets.IntSlider(min=2, max=10, value=5, continuous_update=False),
    w_resistance=widgets.FloatSlider(min=0, max=2, step=0.1, value=1.0, continuous_update=False),
    w_elevation=widgets.FloatSlider(min=0, max=2, step=0.1, value=1.0, continuous_update=False),
    w_valve_cut=widgets.FloatSlider(min=0, max=2, step=0.1, value=1.0, continuous_update=False),
    w_coupling=widgets.FloatSlider(min=0, max=2, step=0.1, value=0.0, continuous_update=False),
    balance_target=widgets.Dropdown(options=['count', 'demand', 'none'], value='count'),
    balance_tol=widgets.FloatSlider(min=0.0, max=0.6, step=0.05, value=0.15,
                                    continuous_update=False),
    seed=widgets.IntSlider(min=0, max=20, value=1, continuous_update=False),
)

'''
ky2 - 4D - 0.59
ky7 - 4D - 0.39
graeme 5D - 0.78
'''

interactive(children=(Dropdown(description='network', index=2, options=('Balerma', 'C_Town', 'D_Town', 'Graeme…

'\nky2 - 4D - 0.59\nky7 - 4D - 0.39\ngraeme 5D - 0.78\n'

## 5. Stress test: contiguity and runtime across the whole corpus

Every network, one partition each, checked for contiguity (must always pass — it's a
verified post-condition, not an assumption) and timed, since the interactive widget above
needs to stay responsive up to ky17's ~6,000 nodes.

In [39]:
import time

stress = []
for net in NETWORKS:
    wn = get_wn(net)
    t0 = time.time()
    try:
        out = di.districts_from_graph(wn, k=6, weights=di.DistrictingWeights(1, 1, 1, 0), seed=2)
        con = di.contiguity_report(out['graph'], out['labels'])
        stress.append({'network': net, 'n_nodes': out['graph'].number_of_nodes(),
                       'seconds': round(time.time() - t0, 2),
                       'fragments_repaired': out['n_fragments_repaired'],
                       'all_contiguous': bool(con.contiguous.all()),
                       'elevation_informative': out['elevation_check']['informative'],
                       'status': 'ok'})
    except Exception as exc:
        stress.append({'network': net, 'seconds': round(time.time() - t0, 2),
                       'status': '%s: %s' % (type(exc).__name__, exc)})
stress = pd.DataFrame(stress)
stress.to_csv(OUT_DIR / 'corpus_stress_test.csv', index=False)
display(stress)
assert stress.query("status == 'ok'").all_contiguous.all(), 'contiguity failed somewhere'
print('\nPASS: every network in the corpus partitions into fully contiguous districts.')
print('slowest: %s at %.1fs' % (stress.loc[stress.seconds.idxmax(), 'network'],
                                stress.seconds.max()))

,network,n_nodes,seconds,fragments_repaired,all_contiguous,elevation_informative,status
0,Balerma,447,0.15,0,True,True,ok
1,C_Town,396,0.11,0,True,True,ok
2,D_Town,407,0.15,0,True,True,ok
3,Graeme,114,0.08,1,True,False,ok
4,NJ1,14999,24.62,1,True,True,ok
5,PA1,339,0.11,0,True,True,ok
6,ky1,859,0.22,0,True,True,ok
7,ky10,935,0.22,0,True,True,ok
8,ky11,831,0.31,0,True,True,ok
9,ky12,2355,1.52,0,True,True,ok



PASS: every network in the corpus partitions into fully contiguous districts.
slowest: NJ1 at 24.6s


## What this establishes, and what remains open

**Established.** A districting method that (1) guarantees contiguity by construction and
verifies it as a post-condition rather than hoping for it, (2) recovers a real utility's
own DMA partition on D-Town (AMI 0.86) from graph structure alone, (3) is honest about a
case where it fails — Graeme, where the cheap proxy (0.05) and the true hydraulic ratio
(0.78) disagree by 14x, and where even the best possible partition of the real coupling
matrix cannot do better, so the network itself has no separable structure to find, and (4)
runs the whole 21-network corpus, including ky17's 6,261 nodes, in seconds.

**Open.**

1. **The coupling factor's coverage is capped by notebook 02's probe count.** On a sparsely
   probed network only a fraction of graph edges get a real coupling value (25.6% on
   Graeme's 60-of-114 probes); the rest fall back to the structural factors regardless of
   `w_coupling`. Raising `max_probe_nodes` upstream, or probing preferentially along the
   candidate cut edges this notebook identifies, would close the gap.
2. **Elevation informativeness must be checked per network, not assumed.** It is checked
   here and the check is enforced (a warning fires in the interactive view), but a
   network-specific default weight profile is not chosen automatically — that judgement is
   left to the person running the notebook.
3. **Balance and separability trade off**, and the trade is not quantified beyond the
   diagnostic pair shown per run. A Pareto sweep over `balance_tol` versus achieved
   coupling ratio would make that trade explicit rather than felt one slider move at a time.
4. **The structural proxy has only one external validation (D-Town) and one warning case
   (Graeme).** A corpus with more real DMA ground truth would strengthen the claim that
   elevation + resistance + valve-respect is a generally reliable proxy rather than one that
   happens to work on a single elevation-organised network.
5. **No leakage, no pressure-dependent demand.** As in the earlier notebooks, real
   distribution loss would couple every node's demand to its own pressure and could shift
   which cuts are genuinely low-coupling.